# XGBoost·CatBoost 평가 지표 및 비교 결론

- 사용 데이터: `weekly_next_week_with_vle.csv`
- Target: `target_next_week_withdrawn`(다음 주 이탈 여부)
    - 동일 학생의 주차별 행이 학습·검증에 동시에 포함되지 않도록 `id_student` 기준 3-Fold GroupKFold를 적용

### 데이터 개요

- 데이터 단위: 학생 1명 × 과목 1개 × 운영회차 1개 × 예측 주차 1개
- 규모: **895,005행 × 53열**, 고유 학생 **26,045명**, 7개 과목, 22개 운영회차, 예측 주차 1~38주
- Target: 다음 주 이탈 6,672건(전체 행의 약 0.7455%)

### 컬럼 구성

1. **식별·운영 정보**: `code_module`, `code_presentation`, `id_student`, `prediction_week`, `cutoff_day`, `module_presentation_length`
2. **학생 기본 특성**: 성별, 지역, 최종 학력, IMD 구간, 연령대, 이전 수강 횟수, 수강 학점, 장애 여부
3. **등록 파생변수**: 등록일을 바탕으로 한 개강 후 등록 여부(`registered_after_start`), 사전 등록일 수(`registration_lead_days`), 지연 등록일 수(`late_registration_days`)
4. **평가 누적 파생변수**: 해당 주차까지의 평가 수·평가 비중·제출 수·지각 제출 수·미제출 수·제출률·평균 점수
5. **VLE 누적 파생변수**: 누적 클릭 수, 상호작용 행 수, 마지막 활동일, 누적 활동일 수, 고유 사이트 수, 활동 유형별 누적 클릭 수, VLE 기록 여부

### 파생변수 생성 원칙

- `prediction_week=w` 행의 입력값은 **w주차 종료일(`cutoff_day = w×7-1`)까지** 관측된 정보만 누적합니다.
- 평가 Feature는 해당 시점까지 마감·제출된 평가만, VLE Feature는 해당 시점까지 발생한 클릭·활동만 집계합니다. 활동 유형별 클릭 수도 같은 기준으로 누적합니다.
- 이탈 학생은 이탈 직전 주차까지만 행을 생성하고, 이탈 후의 0클릭·0활동 행은 만들지 않습니다.
- 미래 주차 VLE·평가 정보, `final_result`, `date_unregistration`, 이탈 주차는 Feature에서 제외해 누수를 방지했습니다.

아래 수치는 세 Fold의 OOF(Out-of-Fold) 예측확률 전체로 계산한 결과입니다.

## 평가 지표 설명

| 지표 | 의미 | 판단 기준 |
|---|---|---|
| `recall_at_top_20pct` | 위험도 상위 20% 학생 안에 실제 이탈자가 얼마나 포함됐는지 | 높을수록 좋음 |
| `pr_auc` | 이탈자(소수)를 일반 학생과 구분하는 성능 | 높을수록 좋음 |
| `brier_score` | 예측확률과 실제 이탈 여부의 오차 | 낮을수록 좋음 |
| `ece_10bin` | 예측확률과 실제 이탈비율의 차이를 10개 구간으로 계산한 확률 보정 오차 | 낮을수록 좋음 |

이탈자는 전체의 약 0.75%로 매우 적기 때문에 Accuracy 대신 PR-AUC를 사용했습니다.
실제 서비스에서는 상위 위험군에 우선 개입하므로 Recall@Top-20%를 함께 확인했고, 화면에 이탈확률을 표시하기 위해 Brier Score와 ECE로 확률 보정 상태도 평가했습니다.

## 지표별 계산 방식

### 1. Recall@Top-20% (`recall_at_top_20pct`)

1. 전체 OOF 예측 행을 이탈 예측확률이 높은 순서로 정렬합니다.
2. 전체 행 수의 상위 20%를 위험군으로 선정합니다. 현재 895,005행이므로 약 179,001행입니다.
3. 그 위험군 안에 포함된 실제 이탈자 수를 전체 실제 이탈자 수로 나눕니다.

`Recall@Top-20% = 상위 위험군 안의 실제 이탈자 수 / 전체 실제 이탈자 수`

예를 들어 XGBoost의 0.6993은 전체 실제 이탈자 6,672명 중 약 4,666명이 예측확률 상위 20% 안에 포함됐다는 의미입니다. 현재 코드는 과목별·주차별로 따로 상위 20%를 뽑는 것이 아니라, 전체 학생×과목×주차 OOF 행에서 상위 20%를 뽑습니다.

### 2. PR-AUC (`pr_auc`)

예측확률 임계값을 바꿔 가며 Precision과 Recall의 관계를 그린 PR 곡선 아래 면적입니다.

- `Precision = 이탈로 예측한 학생 중 실제 이탈자의 비율`
- `Recall = 전체 실제 이탈자 중 모델이 찾아낸 비율`

이탈자가 적은 불균형 데이터에서는 Accuracy보다 PR-AUC가 이탈자 구분 성능을 더 잘 보여줍니다. 값이 높을수록 좋습니다.

### 3. Brier Score (`brier_score`)

각 행의 예측확률과 실제 Target(이탈=1, 비이탈=0)의 차이를 제곱한 뒤 전체 평균을 계산합니다.

`Brier Score = 평균((실제 Target - 예측확률)²)`

예측확률이 실제 결과와 가까울수록 낮아집니다. 즉, 위험군 순위뿐 아니라 ‘이탈확률 30%’라는 수치 자체가 얼마나 정확한지 평가하는 지표입니다.

### 4. ECE 10구간 (`ece_10bin`)

예측확률 0~1을 10개 구간으로 나눕니다. 각 구간에서 평균 예측확률과 실제 이탈비율의 차이를 구하고, 해당 구간 행 비율을 가중치로 적용해 모두 더합니다.

`ECE = Σ(구간 행 비율 × |구간 실제 이탈비율 - 구간 평균 예측확률|)`

예를 들어 예측확률 0.30 부근 학생들의 실제 이탈비율도 0.30에 가까우면 좋은 보정입니다. ECE는 낮을수록 좋으며, 0에 가까울수록 예측확률을 서비스 화면에 신뢰성 있게 표시할 수 있습니다.

## 모델별 전체 평가 결과

| 모델 | Recall@Top-20% ↑ | PR-AUC ↑ | Brier Score ↓ | ECE(10구간) ↓ |
|---|---:|---:|---:|---:|
| XGBoost | **0.6993** | 0.0808 | 0.148658 | 0.299222 |
| CatBoost | 0.6962 | **0.0893** | **0.007066** | **0.000198** |

- XGBoost는 상위 20% 위험군에서 실제 이탈자를 69.93% 포착했습니다.
- CatBoost는 상위 20% 위험군에서 실제 이탈자를 69.62% 포착했습니다.
- 두 모델의 Recall 차이는 약 0.31%p로 작습니다.
- CatBoost는 PR-AUC가 더 높아 이탈자와 비이탈자를 전반적으로 더 잘 구분했습니다.
- CatBoost는 Brier Score와 ECE가 매우 낮아, 예측확률 자체도 실제 이탈비율에 훨씬 가깝습니다.

## Fold별 성능 안정성

| 모델 | Fold 1 PR-AUC | Fold 2 PR-AUC | Fold 3 PR-AUC |
|---|---:|---:|---:|
| XGBoost | 0.0863 | 0.0816 | 0.0802 |
| CatBoost | 0.0919 | 0.0919 | 0.0867 |

CatBoost는 세 Fold 모두에서 XGBoost보다 높은 PR-AUC를 보였고, 성능 변동도 크지 않았습니다.

## 결론

**현재 결과 기준 최종 후보는 CatBoost입니다.**

XGBoost의 Recall@Top-20%가 0.0031 정도 높지만 차이가 매우 작습니다. 반면 CatBoost는 PR-AUC가 더 높고, Brier Score와 ECE가 크게 우수합니다. 따라서 위험군 선별 성능을 유지하면서도 스트림릿 화면에 표시할 이탈확률의 신뢰도가 높은 CatBoost를 우선 채택하는 것이 적절합니다.

단, 최종 배포 전에는 CatBoost에도 Platt Scaling 또는 Isotonic Regression을 적용했을 때 보정 성능이 더 좋아지는지 별도로 검증합니다. XGBoost는 `scale_pos_weight`로 클래스 불균형을 보정했기 때문에 현재의 원시 확률은 그대로 사용자에게 표시하지 않습니다.